In [18]:
from langchain_community.document_loaders import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import json


Setup - Load the CSV file containing the 62 filtered clauses

In [ ]:
## Load the CSV file into a list of document chunks
loader = CSVLoader(file_path="data/addressable_clauses.csv",
                   content_columns=["text"],
                   metadata_columns=["chunk_id", "source"],
                   )
chunks = loader.load()

## Check
print(chunks[0].page_content)
print(chunks[0].metadata)

text: If a project has safety-critical software or mission-critical software, the project manager shall implement the following items in the software:  a. The software is initialized, at first start and restarts, to a known safe state. b. The software safely transitions between all predefined known states. c. Termination performed by software functions is performed to a known safe state. d. Operator overrides of software functions require at least two independent actions by an operator. e. Software rejects commands received out of sequence when execution of those commands out of sequence can cause a hazard. f. The software detects inadvertent memory modification and recovers to a known safe state. g. The software performs integrity checks on inputs and outputs to/from the software system. h. The software performs prerequisite checks prior to the execution of safety-critical software commands. i. No single software event or action is allowed to initiate an identified hazard. j. The soft

In [20]:
##Initialize the Hugging Face embeddings model and create a Chroma vector database from the document chunks
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="addressable_clauses",
    persist_directory="/Users/victorsilva/Downloads/chroma_db"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6121.60it/s]


In [ ]:
## Verify metadata is present in the vector database
results = vector_db.similarity_search("Safety requirements", k=1)
for doc in results:
    print(doc.metadata)
    print(doc.page_content[:100])  # Print the first 100 characters of the content
    print("-----")

{'row': 60, 'chunk_id': 'npr-3.12.1-requirement', 'source': 'NPR 7150.2D'}
text: The project manager shall perform, record, and maintain bi-directional traceability between th
-----


In [22]:
##Test search for a specific query
vector_db.similarity_search("Interface requirements", k=3)

[Document(id='eab080ae-a586-41ce-baba-8aaca0e824ac', metadata={'source': 'SWEHB 5.09', 'chunk_id': 'swehb-3.2.3-guidance', 'row': 43}, page_content='text: Interface Requirements Interface requirements define how the software interacts with other systems, hardware, users, or other software components. These are critical for integration and interoperability as they define the boundaries and communication protocols between the system and its environment. Interface requirements cover all inputs and outputs for the software system to pass data to its appropriate location. There are a multitude of interface types that software systems may need to utilize to fulfill the software’s requirements. Some of the interfaces the software may have include external interfaces, internal interfaces, user interfaces, system interfaces, hardware interfaces, communication interfaces, and interfaces with services. Identify Interface requirements to: Ensure seamless integration with external systems or compon